# CKIP斷詞測試

In [ ]:
import os
from ckipnlp.pipeline import CkipPipeline, CkipDocument
# from ckipnlp.pipeline import CkipPipeline, CkipDocument

pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})

# 导入相关包

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
from collections import Counter
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt

import json
import glob

import jieba
import re
import os

# 停用詞和同義詞list

In [ ]:
import json
# # nuk停用頂端.json (用在斷詞前)
# with open("/content/drive/MyDrive/collab-topic/停用部分句/nuk停用頂端.json", "r", encoding="utf-8") as f:
#     stopwords=json.load(f)
# RemoveTonuk=[]
# for word in stopwords:
#   RemoveTonuk=set(word.get("stopwords", []))# 用set防止重複的東西
# # print("toRemove:\n",toRemove)

# # nuch停用頂端.json (用在斷詞前)
# with open("/content/drive/MyDrive/collab-topic/停用部分句/nchu停用頂端.json", "r", encoding="utf-8") as f:
#     stopwords=json.load(f)
# RemoveTonuchu=[]
# for word in stopwords:
#   RemoveTonuchu=set(word.get("stopwords", []))# 用set防止重複的東西

# 無主題式詞 (用在特徵選取後)
with open("./stopWords/meaningless.json", "r", encoding="utf-8") as f:
    stopwords=json.load(f)
RemoveTheMeaningless=[]
for word in stopwords:
  RemoveTheMeaningless=set(word.get("stopwords", []))# 用set防止重複的東西

# 同義詞清單
synonymsDocument="./stopWords/synonyms.json"
# 讀取同義詞JSON檔案 key:arraylist
with open(synonymsDocument, 'r', encoding='utf-8') as f:
    synonyms = json.load(f)
# print(synonyms)
# 只准許的詞性
valid_pos = {"Na", "Nb", "Nc", "Nv", "VA", "VAC", "VB", "VC", "VCL", "VD", "VE", "VF", "VG", "VH", "VHC", "VI", "VJ", "VK", "VL"}
BanSthwithEntity_Types=['PERSON','LOC','GPE']
HFW_rank=10 # High Frequency Word only rank to 10 = 關鍵字 = (經特徵選取後的)特徵字

# 相似度計算

## 新文本預處理

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from collections import Counter
import re
# 預編譯正則表達式
replace_pattern = re.compile(r'^(?=.{2,}).*盃$')
remove_pattern = re.compile(r'^(?!会|人|金枝演)(^.+社$)$')

# pipeline = CkipPipeline(opts={'con_parser': {'username': "<CKIP_USERNAME>", 'password': "<CKIP_PASSWORD>"}})
# pipeline = CkipPipeline()  # create a CKIP pipeline object
def preprocess_text(text):
    # 刪除英文 CKIP搞不定外文，還會結合在一起標FW
    text = re.sub(r'[a-zA-Z]', ' ', text)
    doc = CkipDocument(raw=text) # 將一段文字轉換成 CkipTagger 能夠處理的文件形式
    ws_text=pipeline.get_ws(doc)
    pos_text=pipeline.get_pos(doc)
     # 作實體識別(用於判別人名)
    ner_text = pipeline.get_ner(doc)
    # 只選特定詞性(同時篩掉停用詞)
    featureWord=[]
    for i,sent_p in enumerate(pos_text):
      for j,word_p in enumerate(sent_p):
        # if ner_text[i][j]=='PERSON':
        #   continue
        if word_p in valid_pos and ws_text[i][j] not in RemoveTheMeaningless:
            featureWord.append(ws_text[i][j])
    
    # 替換同義詞
    item_text = ' '.join(featureWord)
    for key, values in synonyms.items():
        for value in values:
            item_text = item_text.replace(value, key)
    featureWord = item_text.split()
    # 使用預編譯正則表達式
    featureWord = [replace_pattern.sub('比賽', c) for c in featureWord]
    featureWord = [remove_pattern.sub('社團', c) for c in featureWord]

    # 禁人名/LOC/GPE出現在featureWord的清單(list-banSth)
    banSth=[] # PersonName or LOC or GPE
    # print("ner_text\n",ner_text)
    for i, sent_ner in enumerate(ner_text):
      # print("sent_ner\n",sent_ner)
      for j, ner in enumerate(sent_ner):
        # print("ner\n",ner)
        # print("ner[1]\n",ner[1])
        if ner[1] in BanSthwithEntity_Types:
          banSth.append(ner[0])
          # print("ner\n",ner)
    # 只留中文字
    newfeatureWord = []
    pattern = re.compile(r'[\u4e00-\u9fff]+')
    for word in featureWord:
      # 把列表中的每個單詞轉成字符串再進行匹配
      matched_words = pattern.findall(str(word))# 我覺得跟str有關
      # 把匹配到的單詞加入到新列表中
      newfeatureWord.extend(matched_words)
    featureWord=newfeatureWord

    # 禁在清單內的東西
    if len(banSth):# 有東西，長度不為0，執行；有人名被禁，禁人名清單長度不為0，執行
      featureWord=[w for w in featureWord if w not in banSth]
    # # 檢查關鍵字用
    # if len(featureWord)>20:
    #   print(f"In {counterFor} featureWord is many reaching {len(featureWord)}\n",featureWord)
    # # 無意義文本不放
    # if len(featureWord)==0:
    #   print(f"In {counterFor} of {json_file} featureWord is Error",)
    #   print(f"Error: featureWord is empty",)
    return featureWord
    
new_text = '測試測試這是文本this is a new document for topic prediction 搜尋活動'
print(new_text)

# 預處理新文本，建立詞彙表
new_doc = preprocess_text(new_text)
print(new_doc)

## 轉向量得相似度(優化版)

In [ ]:
# ___7最短時間
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json
import time

start_time = time.time()

# 新文本
# new_text = '''
# 教學助理工作坊---輕鬆學會高效雲端筆記法工作坊
# ❀ 課程大綱 ❀
#  基本內容操作
# 筆記本介面與內容操作
# 內容新增與擷取方式
# 內容標註與連結
#  應用操作
# 記事方式應用
# 多媒體記事
# 多載具應用
# '''
# new_text ='''"【轉知校外活動】國立中山大學為提升區域內大學教師全英語授課能力，訂於112年1月7日(六)辦理「EMI教師培訓計畫-進階工作坊(第2、3場)」,歡迎本校有興趣之教師踴躍報名參加! ",
#     "url": "https://cetlr.nuk.edu.tw/p/406-1060-59554,r1734.php?Lang=zh-tw",
#     "\n\n中心簡介\n中心成員\n活動資訊中心活動校外活動\n法規及表單\n活動剪影",
#         "\n\n回首頁\n高大首頁\n網站導覽\nENGLISH",
#         "\n\n 搜尋 ",
#         "\n ",
#         "\n首頁\n校外活動",
#         "\n\n【轉知校外活動】國立中山大學為提升區域內大學教師全英語授課能力，訂於112年1月7日(六)辦理「EMI教師培訓計畫-進階工作坊(第2、3場)」,歡迎本校有興趣之教師踴躍報名參加!",
#         "\n說明：\n\n一、國立中山大學獲教育部補助「大專校院學生雙語化學習計畫」重點培育學校及獲核成立「南區雙語教育區域資源中心」，為發揮重點培育學校資源共享效益，並提升大學教師全英語授課能量，訂於112年1月7日(六)舉辦EMI教師培訓進階課程(第2、3場)。\n二、課程訊息：\n(一)日期：112年1月7日(六)8:30-16:30。\n(二)地點：國立中山大學圖資大樓10樓SW1005創新教室C。\n(三)人數：上限60名。\n(四)報名方式：線上報名，網址： https://reurl.cc/MXNN33 【12月30日(五)17:00截止報名】。\n三、檢附活動海報及交通資訊各乙份。本案聯絡人：蔡仲茵小姐，電話：(07)525-2000分機2164，redacted-user@example.com。",
#         "\n\n瀏覽數:\n\n友善列印\n分享",
#         "\n\n ",
#         "\n "
#     '''
# 新文本特徵選取
new_text='''
 "\n\n 搜尋 ",
        "\n ",
        "\n首頁",
        "\n\n【轉知校外活動】銘傳大學應用英語學系辦理2023年「國際應用英語教學研討會暨工作坊」,歡迎本校所屬之師生踴躍報名參加!",
        "\n說明：\n\n一、旨揭研討會會議主題: Exploring English Education in the Post-Pandemic Era: Reflection and Prospect 「探究後疫情時代的英語教育:省思與展望」。\n二、會議日期:112年3月17日至18日(五~六)。\n三、會議地點:銘傳大學基河校區(臺北市基河路 130 號)。\n四、報名資訊:https://forms.gle/65cQ6RiEoigf8ygH6。\n五、檢附研討會會議資訊乙份。\n六、聯絡人:郭棨鈞助理，03-350-7001分機3211-3213。",
        "\n附件_研討會暨工作坊會議資訊.pdf",
        "\n\n瀏覽數:\n\n友善列印\n分享",
        "\n\n ",
        "\n "
    ],
    "time": "",
    "title": "【轉知校外活動】銘傳大學應用英語學系辦理2023年「國際應用英語教學研討會暨工作坊」,歡迎本校所屬之師生踴躍報名參加! ",
    "url": "https://cetlr.nuk.edu.tw/p/404-1060-61321.php?Lang=zh-tw"
'''
# str(new_text)
new_text = preprocess_text(new_text)
new_text_feature=[' '.join(new_text)]
print(new_text_feature)

# (被比較的)其他文本清单
MyTextListPath = './featureWords/nuk+nchu+ntnu+ntust+nyust+0228nkust+govFromOpenData.json'
with open(MyTextListPath, 'r', encoding='utf-8') as f:
    other_texts = json.load(f)

# 将所有文本合并为一个文本列表
all_text = [' '.join(text['TitleAndDESC']) for text in other_texts]

# 使用 CountVectorizer 将文本转换为特征矩阵
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(all_text)

# 新文本的特征矩阵
new_text_vector = vectorizer.transform(new_text_feature)

# 遍历所有文本，计算每个文本与新文本的余弦相似度
sim_scores = []
# counterTest=0
for i, text in enumerate(other_texts):
    # 測試
    # if counterTest==1: break
    # counterTest+=1
    old_text_vector = X.getrow(i)  # 获取第 i 个文本的特征向量
    # print("old_text_vectori\n",old_text_vector)

    # Iold_text=[' '.join(text['TitleAndDESC'])]
    # # 其他文本清单的個別文本的詞向量
    # vector = vectorizer.transform(Iold_text)# 對text_dict['TitleAndDESC']適應all_text的詞向量
    # print("vector\n",vector)

    sim_score = cosine_similarity(old_text_vector, new_text_vector)[0][0]
    sim_scores.append((text["EventID"], sim_score))

    # # 添加一个中断条件，计算相似度的文本数量
    # if i % 10000 == 0:
    #     print(f"{i} texts have been processed in {time.time() - start_time:.2f} seconds.")

# 将相似度从高到低排序
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
# sorted_event_ids = [event_id for event_id, sim_score in sorted(sim_scores, key=lambda x: x[1], reverse=True)]

print(f"texts have been processed in {time.time() - start_time:.2f} seconds.")# 8.63 seconds

# # 打印相似度最高的前10个文本
# for i, (event_id, sim_score) in enumerate(sim_scores[:10]):
#     print(f"{i+1}. Event ID: {event_id}, Similarity Score: {sim_score:.4f}")
print(sim_scores)
# print(sorted_event_ids)

在此代码中，将TitleAndDESC中的所有文本合并为一个文本列表是**为了将它们合并为一个大的语料库**，从而可以在CountVectorizer中使用该语料库来计算文本的向量表示形式，从而进行余弦相似度计算。这个过程通常称为词频统计或词袋模型，它通过将每个文本视为词汇表中每个单词的出现次数来表示它。 由于向量的长度是由词汇表中单词的数量决定的，因此将所有文本合并为一个大列表可以确保向量的长度相同，从而使计算余弦相似度更容易。

(0, 4) 1 表示第0个词语在第4个文本中出现了1次，(1, 2) 1 表示第1个词语在第2个文本中出现了1次，依此类推。

similarity是一個Python模組，提供用於計算文本相似度的類和函數。其中，compute是類similarity中的一個方法，用於計算兩個文本的相似度。

以下是compute方法的所有參數和說明：

text1（必填）：第一個文本字符串，用於計算相似度。
text2（必填）：第二個文本字符串，用於計算相似度。
method（選填）：計算相似度的方法。預設值為cosine，表示使用余弦相似度來計算。其他可用的方法有jaccard（Jaccard相似度）和levenshtein（Levenshtein距離）。
remove_stopwords（選填）：是否移除停用詞。預設值為True，表示移除停用詞。如果設置為False，則不會移除停用詞。
stopwords_language（選填）：停用詞列表的語言。預設值為english，表示使用英文停用詞列表。其他可用的語言有chinese（中文停用詞列表）和french（法文停用詞列表）等。
lemmatize（選填）：是否將詞形還原為詞根。預設值為True，表示進行詞形還原。如果設置為False，則不進行詞形還原。
在上述代碼中，首先使用pip install simtext安裝了simtext模組，然後創建了一個similarity類的實例sim。接著，使用compute方法計算了兩個文本text1和text2之間的相似度，並將結果存儲在變量res中。最後，使用print函數印出了計算結果。由於預設使用的是余弦相似度，因此計算結果是一個介於0和1之間的浮點數，越接近1表示兩個文本越相似。